# 03 — Data Cleaning & Preparation

## Objective

Prepare eight insurance datasets for reliable SQL analysis,
Python analytics, and Power BI reporting while preserving
the integrity of the original data.

## Cleaning Strategy

- Standardize column data types and date formats.
- Preserve original financial values and policy statuses.
- Create data-quality flags for identified business-rule inconsistencies.
- Retain legitimate statistical outliers.
- Prepare consistent datasets for downstream analysis.




In [1]:
# Step 3.2 — Import Libraries and Load Insurance Datasets

import pandas as pd
import numpy as np
from pathlib import Path

# Define the original data directory
data_path = Path("../data")

# Load all eight insurance datasets
datasets = {
    "Customers": pd.read_csv(data_path / "customers.csv"),
    "Policies": pd.read_csv(data_path / "policies.csv"),
    "Claims": pd.read_csv(data_path / "claims.csv"),
    "Payments": pd.read_csv(data_path / "payments.csv"),
    "Risk Assessment": pd.read_csv(data_path / "risk_assessment.csv"),
    "Agents": pd.read_csv(data_path / "agents.csv"),
    "Policy Agents": pd.read_csv(data_path / "policy_agents.csv"),
    "Vehicles": pd.read_csv(data_path / "vehicles.csv")
}

# Define the analysis reference date
reference_date = pd.Timestamp("2026-09-30")

# Verify successful loading
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns)
    }
    for name, df in datasets.items()
])

print("Insurance Datasets Loaded Successfully")
display(dataset_summary)

print("\nTotal Records:", dataset_summary["Rows"].sum())

Insurance Datasets Loaded Successfully


,Dataset,Rows,Columns
0,Customers,10000,6
1,Policies,15000,8
2,Claims,25000,7
3,Payments,15541,5
4,Risk Assessment,10000,4
5,Agents,250,4
6,Policy Agents,15000,2
7,Vehicles,6000,5



Total Records: 96791


In [2]:
# Step 3.3 — Standardize Date Columns Across Insurance Datasets

# Define date columns for each dataset
date_columns = {
    "Customers": ["Join_Date"],
    "Policies": ["Start_Date", "End_Date"],
    "Claims": ["Claim_Date"],
    "Payments": ["Payment_Date"],
    "Risk Assessment": ["Assessment_Date"]
}

# Convert date columns to datetime
for dataset_name, columns in date_columns.items():

    for column in columns:

        datasets[dataset_name][column] = pd.to_datetime(
            datasets[dataset_name][column],
            errors="coerce"
        )

# Verify date column data types
date_type_summary = []

for dataset_name, columns in date_columns.items():

    for column in columns:

        date_type_summary.append({
            "Dataset": dataset_name,
            "Column": column,
            "Data_Type": str(datasets[dataset_name][column].dtype),
            "Invalid_Dates": int(
                datasets[dataset_name][column].isna().sum()
            )
        })

date_type_summary = pd.DataFrame(date_type_summary)

print("Date Standardization Summary:")
display(date_type_summary)

print(
    "\nTotal Invalid Dates:",
    date_type_summary["Invalid_Dates"].sum()
)

Date Standardization Summary:


,Dataset,Column,Data_Type,Invalid_Dates
0,Customers,Join_Date,datetime64[us],0
1,Policies,Start_Date,datetime64[us],0
2,Policies,End_Date,datetime64[us],0
3,Claims,Claim_Date,datetime64[us],0
4,Payments,Payment_Date,datetime64[us],0
5,Risk Assessment,Assessment_Date,datetime64[us],0



Total Invalid Dates: 0


In [3]:
# Step 3.4 — Standardize Categorical Columns Across Datasets

# Define categorical columns
categorical_columns = {
    "Customers": ["Gender", "Region"],
    "Policies": ["Policy_Type", "Policy_Status"],
    "Claims": ["Claim_Status", "Claim_Type"],
    "Payments": ["Payment_Method"],
    "Risk Assessment": ["Risk_Category"],
    "Agents": ["Region", "Channel"],
    "Vehicles": ["Vehicle_Type"]
}

# Standardize whitespace in categorical values
for dataset_name, columns in categorical_columns.items():

    for column in columns:

        datasets[dataset_name][column] = (
            datasets[dataset_name][column]
            .astype("string")
            .str.strip()
        )

# Verify standardized categories
category_summary = []

for dataset_name, columns in categorical_columns.items():

    for column in columns:

        values = datasets[dataset_name][column]

        category_summary.append({
            "Dataset": dataset_name,
            "Column": column,
            "Unique_Categories": values.nunique(),
            "Missing_Values": int(values.isna().sum()),
            "Blank_Values": int(values.eq("").sum())
        })

category_summary = pd.DataFrame(category_summary)

print("Categorical Standardization Summary:")
display(category_summary)

print("\nTotal Missing Values:",
      category_summary["Missing_Values"].sum())

print("Total Blank Values:",
      category_summary["Blank_Values"].sum())

Categorical Standardization Summary:


,Dataset,Column,Unique_Categories,Missing_Values,Blank_Values
0,Customers,Gender,3,0,0
1,Customers,Region,5,0,0
2,Policies,Policy_Type,4,0,0
3,Policies,Policy_Status,3,0,0
4,Claims,Claim_Status,4,0,0
5,Claims,Claim_Type,4,0,0
6,Payments,Payment_Method,3,0,0
7,Risk Assessment,Risk_Category,3,0,0
8,Agents,Region,5,0,0
9,Agents,Channel,3,0,0



Total Missing Values: 0
Total Blank Values: 0


In [4]:
# Step 3.5 — Standardize Numerical Data Types

# Define numerical columns for each dataset
numerical_columns = {
    "Customers": ["Age", "Annual_Income"],
    "Policies": ["Annual_Premium", "Coverage_Amount"],
    "Claims": ["Claim_Amount", "Fraud_Flag"],
    "Payments": ["Payment_Amount"],
    "Risk Assessment": ["Risk_Score"],
    "Agents": ["Experience_Years"],
    "Vehicles": ["Vehicle_Age", "Insured_Value"]
}

# Define columns that should contain integers
integer_columns = {
    "Age",
    "Fraud_Flag",
    "Risk_Score",
    "Experience_Years",
    "Vehicle_Age"
}

numerical_summary = []

for dataset_name, columns in numerical_columns.items():

    for column in columns:

        # Convert to numerical values
        converted = pd.to_numeric(
            datasets[dataset_name][column],
            errors="coerce"
        )

        # Apply appropriate data type
        if column in integer_columns:
            datasets[dataset_name][column] = converted.astype("Int64")
        else:
            datasets[dataset_name][column] = converted.astype("float64")

        numerical_summary.append({
            "Dataset": dataset_name,
            "Column": column,
            "Data_Type": str(datasets[dataset_name][column].dtype),
            "Missing_Values": int(
                datasets[dataset_name][column].isna().sum()
            )
        })

# Display validation results
numerical_type_summary = pd.DataFrame(numerical_summary)

print("Numerical Data Type Standardization Summary:")
display(numerical_type_summary)

print(
    "\nTotal Missing Numerical Values:",
    numerical_type_summary["Missing_Values"].sum()
)

Numerical Data Type Standardization Summary:


,Dataset,Column,Data_Type,Missing_Values
0,Customers,Age,Int64,0
1,Customers,Annual_Income,float64,0
2,Policies,Annual_Premium,float64,0
3,Policies,Coverage_Amount,float64,0
4,Claims,Claim_Amount,float64,0
5,Claims,Fraud_Flag,Int64,0
6,Payments,Payment_Amount,float64,0
7,Risk Assessment,Risk_Score,Int64,0
8,Agents,Experience_Years,Int64,0
9,Vehicles,Vehicle_Age,Int64,0



Total Missing Numerical Values: 0


## Policy Data Quality Flags

### Objective
Identify policy-level business-rule inconsistencies without modifying or deleting original records.

### Methodology
- Flag policies starting before the customer's recorded joining date.
- Derive coverage status using the analysis reference date of September 30, 2026.
- Identify inconsistencies between recorded policy status and coverage dates.
- Preserve the original `Policy_Status` for comparison and reporting.

### Business Considerations
The derived coverage status represents the policy's date-based coverage period, not its confirmed administrative or contractual status.

Flagged records will be retained for investigation and downstream analysis.

In [5]:
# Step 3.6 — Create Policy-Level Data Quality Flags
# Preserve original policy records and identify business-rule discrepancies.

# Create working references
policies = datasets["Policies"]
customers = datasets["Customers"]

# Map customer joining dates to policies
customer_join_dates = customers.set_index("Customer_ID")["Join_Date"]

policies["Customer_Join_Date"] = (
    policies["Customer_ID"].map(customer_join_dates)
)

# Flag policies starting before the customer's recorded joining date
policies["Flag_Start_Before_Join"] = (
    policies["Start_Date"] < policies["Customer_Join_Date"]
).astype("Int64")

# Derive coverage status as of the analysis reference date
policies["Coverage_Status_As_Of"] = np.select(
    [
        policies["Start_Date"] > reference_date,
        policies["End_Date"] < reference_date
    ],
    [
        "Not Yet Started",
        "Expired"
    ],
    default="In Coverage Period"
)

# Flag recorded Active policies with an expired coverage period
policies["Flag_Active_After_Expiry"] = (
    (policies["Policy_Status"] == "Active") &
    (policies["End_Date"] < reference_date)
).astype("Int64")

# Flag recorded Expired policies with an end date on/after reference date
policies["Flag_Expired_Before_End"] = (
    (policies["Policy_Status"] == "Expired") &
    (policies["End_Date"] >= reference_date)
).astype("Int64")

# Summarize policy data quality flags
policy_flag_summary = pd.DataFrame({
    "Validation": [
        "Policies Starting Before Customer Join Date",
        "Active Policies Past End Date",
        "Expired Policies Before End Date",
        "Total Policy Status Discrepancies"
    ],
    "Flagged_Records": [
        policies["Flag_Start_Before_Join"].sum(),
        policies["Flag_Active_After_Expiry"].sum(),
        policies["Flag_Expired_Before_End"].sum(),
        (
            policies["Flag_Active_After_Expiry"].sum()
            + policies["Flag_Expired_Before_End"].sum()
        )
    ]
})

print("Policy Data Quality Flag Summary:")
display(policy_flag_summary)

print("\nDerived Coverage Status:")
display(policies["Coverage_Status_As_Of"].value_counts())

Policy Data Quality Flag Summary:


,Validation,Flagged_Records
0,Policies Starting Before Customer Join Date,3512
1,Active Policies Past End Date,7306
2,Expired Policies Before End Date,618
3,Total Policy Status Discrepancies,7924



Derived Coverage Status:


Coverage_Status_As_Of
Expired               13224
In Coverage Period     1776
Name: count, dtype: int64

### Key Findings 
- 3,512 policies started before their customer's recorded joining date.
- 7,924 policies have discrepancies between recorded status and coverage dates.
- As of September 30, 2026, 13,224 policies were past their end date, while 1,776 were within their coverage period.
- Original policy statuses and financial values were preserved.

**Retain all flagged records and use the derived coverage status separately from the original policy status.**

In [6]:
# Step 3.7 — Create Claim-Level Coverage Flags

claims = datasets["Claims"]
policies = datasets["Policies"]

# Map policy coverage amounts to claims
policy_coverage = policies.set_index("Policy_ID")["Coverage_Amount"]

claims["Policy_Coverage_Amount"] = (
    claims["Policy_ID"].map(policy_coverage)
)

# Flag claims exceeding associated policy coverage
claims["Flag_Claim_Exceeds_Coverage"] = (
    claims["Claim_Amount"] > claims["Policy_Coverage_Amount"]
).astype("Int64")

# Summarize flagged claims
claim_coverage_summary = pd.DataFrame({
    "Metric": [
        "Total Claims",
        "Claims Exceeding Coverage",
        "Claims Within Coverage"
    ],
    "Count": [
        len(claims),
        int(claims["Flag_Claim_Exceeds_Coverage"].sum()),
        int((claims["Flag_Claim_Exceeds_Coverage"] == 0).sum())
    ]
})

print("Claim Coverage Validation Summary:")
display(claim_coverage_summary)

Claim Coverage Validation Summary:


,Metric,Count
0,Total Claims,25000
1,Claims Exceeding Coverage,143
2,Claims Within Coverage,24857


### Key Finding — Claim Coverage Validation

Out of 25,000 claims, 143 (0.57%) exceed their associated
policy coverage amount.

These claims are flagged for further investigation rather
than automatically adjusted or removed.

The flag identifies a potential coverage discrepancy;
it does not confirm that a claim is invalid.

In [7]:
# Step 3.8 — Create Payment-Level Data Quality Flags

payments = datasets["Payments"]
claims = datasets["Claims"]

# Map each payment to its associated policy
claim_policy_map = claims.set_index("Claim_ID")["Policy_ID"]

payments["Policy_ID"] = (
    payments["Claim_ID"].map(claim_policy_map)
)

# Map policy coverage amounts
policy_coverage_map = (
    datasets["Policies"]
    .set_index("Policy_ID")["Coverage_Amount"]
)

payments["Policy_Coverage_Amount"] = (
    payments["Policy_ID"].map(policy_coverage_map)
)

# Flag payments exceeding associated policy coverage
payments["Flag_Payment_Exceeds_Coverage"] = (
    payments["Payment_Amount"] > payments["Policy_Coverage_Amount"]
).astype("Int64")

# Flag payments after the analysis cutoff
payments["Flag_Payment_After_Cutoff"] = (
    payments["Payment_Date"] > reference_date
).astype("Int64")

# Summarize payment quality flags
payment_flag_summary = pd.DataFrame({
    "Validation": [
        "Total Payments",
        "Payments Exceeding Policy Coverage",
        "Payments After Analysis Cutoff"
    ],
    "Count": [
        len(payments),
        int(payments["Flag_Payment_Exceeds_Coverage"].sum()),
        int(payments["Flag_Payment_After_Cutoff"].sum())
    ]
})

print("Payment Data Quality Flag Summary:")
display(payment_flag_summary)

# Calculate the value of payments after the cutoff
future_payment_amount = payments.loc[
    payments["Flag_Payment_After_Cutoff"] == 1,
    "Payment_Amount"
].sum()

print(
    "\nPayment Amount After Cutoff:",
    f"{future_payment_amount:,.2f}"
)

Payment Data Quality Flag Summary:


,Validation,Count
0,Total Payments,15541
1,Payments Exceeding Policy Coverage,34
2,Payments After Analysis Cutoff,189



Payment Amount After Cutoff: 6,137,939.33


### Key Findings — Payment Data Quality

- 34 payments exceed their associated policy coverage amount.
- 189 payments totaling 6,137,939.33 were recorded after the September 30, 2026 analysis cutoff.
- All original payment records and amounts were preserved.

### Analytical Decision

Payments after the analysis cutoff will be excluded from
historical financial KPIs calculated as of September 30, 2026.

These payments will remain available for data quality
investigation and future-period analysis.

Payments exceeding coverage are flagged for investigation,
not automatically classified as invalid.

In [8]:
# Step 3.9 — Flag Settled Claims Without Payment Records

claims = datasets["Claims"]
payments = datasets["Payments"]

# Identify claims with at least one payment record
paid_claim_ids = set(payments["Claim_ID"])

# Flag settled claims without recorded payments
claims["Flag_Settled_Without_Payment"] = (
    (claims["Claim_Status"] == "Settled") &
    (~claims["Claim_ID"].isin(paid_claim_ids))
).astype("Int64")

# Calculate flagged claim count and amount
settled_without_payment = claims[
    claims["Flag_Settled_Without_Payment"] == 1
]

settlement_summary = pd.DataFrame({
    "Metric": [
        "Total Claims",
        "Settled Claims Without Payment",
        "Claim Amount of Flagged Settled Claims"
    ],
    "Value": [
        len(claims),
        len(settled_without_payment),
        round(settled_without_payment["Claim_Amount"].sum(), 2)
    ]
})

print("Settlement Payment Validation Summary:")
display(settlement_summary)

Settlement Payment Validation Summary:


,Metric,Value
0,Total Claims,25000.0
1,Settled Claims Without Payment,732.0
2,Claim Amount of Flagged Settled Claims,31687328.9


### Key Findings — Settlement Payment Validation

- Identified 732 settled claims without matching payment records.
- These claims represent a combined claim amount of 31,687,328.90.
- All original claim statuses and payment records were preserved.



**Flagged claims will be retained for settlement and payment
performance analysis.

** The absence of a payment record does not necessarily indicate
an unpaid claim or outstanding financial liability. 

In [9]:
# Step 3.10 — Identify Claim Amount Outliers Using IQR

claims = datasets["Claims"]

# Calculate quartiles and interquartile range
q1 = claims["Claim_Amount"].quantile(0.25)
q3 = claims["Claim_Amount"].quantile(0.75)

iqr = q3 - q1

# Calculate outlier boundaries
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Flag claim amount outliers
claims["Flag_Claim_Amount_Outlier"] = (
    (claims["Claim_Amount"] < lower_bound) |
    (claims["Claim_Amount"] > upper_bound)
).astype("Int64")

# Summarize results
claim_outlier_summary = pd.DataFrame({
    "Metric": [
        "Total Claims",
        "Claim Amount Outliers",
        "Claim Amount Non-Outliers",
        "Outlier Percentage"
    ],
    "Value": [
        len(claims),
        int(claims["Flag_Claim_Amount_Outlier"].sum()),
        int((claims["Flag_Claim_Amount_Outlier"] == 0).sum()),
        round(claims["Flag_Claim_Amount_Outlier"].mean() * 100, 2)
    ]
})

print("Claim Amount Outlier Summary:")
display(claim_outlier_summary)

print(f"\nLower Bound: {lower_bound:,.2f}")
print(f"Upper Bound: {upper_bound:,.2f}")

Claim Amount Outlier Summary:


,Metric,Value
0,Total Claims,25000.00
1,Claim Amount Outliers,1583.00
2,Claim Amount Non-Outliers,23417.00
3,Outlier Percentage,6.33



Lower Bound: -36,780.79
Upper Bound: 106,271.22


In [10]:
# Step 3.10.1 — Verify Claim Amount IQR Outlier Calculation

claim_amounts = datasets["Claims"]["Claim_Amount"]

# Recalculate quartiles and boundaries
q1_check = claim_amounts.quantile(0.25)
q3_check = claim_amounts.quantile(0.75)

iqr_check = q3_check - q1_check

lower_check = q1_check - 1.5 * iqr_check
upper_check = q3_check + 1.5 * iqr_check

# Count outliers using both boundaries
lower_outliers = (claim_amounts < lower_check).sum()
upper_outliers = (claim_amounts > upper_check).sum()

print(f"Q1: {q1_check:.4f}")
print(f"Q3: {q3_check:.4f}")
print(f"IQR: {iqr_check:.4f}")

print(f"\nLower Bound: {lower_check:.4f}")
print(f"Upper Bound: {upper_check:.4f}")

print(f"\nLower Outliers: {lower_outliers}")
print(f"Upper Outliers: {upper_outliers}")
print(f"Total Outliers: {lower_outliers + upper_outliers}")

print(f"\nMissing Claim Amounts: {claim_amounts.isna().sum()}")

Q1: 16863.7125
Q3: 52626.7175
IQR: 35763.0050

Lower Bound: -36780.7950
Upper Bound: 106271.2250

Lower Outliers: 0
Upper Outliers: 1583
Total Outliers: 1583

Missing Claim Amounts: 0


### Methodology — Claim Amount Outlier Detection

The Interquartile Range (IQR) method was applied to identify
unusually high or low claim amounts.

**Formula:**

$$
IQR = Q_3 - Q_1
$$

$$
Lower\ Bound = Q_1 - 1.5 \times IQR
$$

$$
Upper\ Bound = Q_3 + 1.5 \times IQR
$$

### Key Findings

- Q1: 16,863.7125
- Q3: 52,626.7175
- IQR: 35,763.0050
- Lower bound: -36,780.7950
- Upper bound: 106,271.2250
- Identified 1,583 claim amount outliers (6.33%).


All outliers are retained because unusually large claims
may represent legitimate high-severity insurance events.

The outlier flag will support claim severity and risk analysis.

**Validation Note:** The current count differs by two records
from the earlier audit and requires reconciliation.

In [11]:
# Step 3.10.2 — Check Outlier Counts Under Alternative Quartile Methods

claim_amounts = datasets["Claims"]["Claim_Amount"]

for method in ["linear", "lower", "higher", "midpoint", "nearest"]:

    q1 = claim_amounts.quantile(0.25, interpolation=method)
    q3 = claim_amounts.quantile(0.75, interpolation=method)

    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_count = (
        (claim_amounts < lower) |
        (claim_amounts > upper)
    ).sum()

    print(f"{method}: {outlier_count} outliers")

linear: 1583 outliers
lower: 1583 outliers
higher: 1583 outliers
midpoint: 1583 outliers
nearest: 1583 outliers


In [12]:
# Step 3.11 — Identify Payment Amount Outliers Using IQR

payments = datasets["Payments"]

# Calculate quartiles
q1 = payments["Payment_Amount"].quantile(0.25)
q3 = payments["Payment_Amount"].quantile(0.75)

# Calculate IQR and boundaries
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Flag payment amount outliers
payments["Flag_Payment_Amount_Outlier"] = (
    (payments["Payment_Amount"] < lower_bound) |
    (payments["Payment_Amount"] > upper_bound)
).astype(int)

# Display results
print("Total Payments:", len(payments))

print(
    "Payment Amount Outliers:",
    payments["Flag_Payment_Amount_Outlier"].sum()
)

print(
    "Outlier Percentage:",
    round(payments["Flag_Payment_Amount_Outlier"].mean() * 100, 2)
)

print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))

Total Payments: 15541
Payment Amount Outliers: 1024
Outlier Percentage: 6.59
Lower Bound: -28812.76
Upper Bound: 82024.08


### Key Findings — Payment Amount Outliers

- Identified 1,024 payment amount outliers (6.59%) using the IQR method.
- The upper outlier threshold is 82,024.08.
- All payment amounts were preserved without modification.
- The current result differs by four records from the earlier audit.

### Analytical Decision

Outliers are retained because unusually high payments may
represent legitimate high-value insurance settlements.

The outlier flag will support payment performance and
financial risk analysis.

In [13]:
# Step 3.12 — Create Overall Claim Review Flag

claims = datasets["Claims"]

# Flag claims requiring further investigation
claims["Flag_Requires_Review"] = (
    (claims["Flag_Claim_Exceeds_Coverage"] == 1) |
    (claims["Flag_Settled_Without_Payment"] == 1) |
    (claims["Flag_Claim_Amount_Outlier"] == 1)
).astype(int)

# Display results
print("Total Claims:", len(claims))

print(
    "Claims Requiring Review:",
    claims["Flag_Requires_Review"].sum()
)

print(
    "Claims Not Flagged:",
    (claims["Flag_Requires_Review"] == 0).sum()
)

print(
    "Review Percentage:",
    round(claims["Flag_Requires_Review"].mean() * 100, 2)
)

Total Claims: 25000
Claims Requiring Review: 2306
Claims Not Flagged: 22694
Review Percentage: 9.22


### Key Findings — Overall Claim Review

- 2,306 claims (9.22%) were flagged for further review.
- 22,694 claims (90.78%) were not flagged by the three selected checks.
- The combined flag considers coverage discrepancies, settled claims without payment records, and claim amount outliers.

### Analytical Decision

The review flag will support claim quality monitoring and
prioritization in Power BI.

Flagged claims are not automatically fraudulent or invalid.
All original records are retained.

In [14]:
# Step 3.13 — Validate Cleaned Insurance Datasets

# Original dataset row counts
expected_rows = {
    "Customers": 10000,
    "Policies": 15000,
    "Claims": 25000,
    "Payments": 15541,
    "Risk Assessment": 10000,
    "Agents": 250,
    "Policy Agents": 15000,
    "Vehicles": 6000
}

# Validate row counts and missing values
validation_results = []

for name, df in datasets.items():

    row_count = len(df)
    missing_values = df.isna().sum().sum()

    status = "PASS" if (
        row_count == expected_rows[name]
        and missing_values == 0
    ) else "REVIEW"

    validation_results.append({
        "Dataset": name,
        "Rows": row_count,
        "Missing_Values": missing_values,
        "Status": status
    })

validation_summary = pd.DataFrame(validation_results)

print("Final Data Cleaning Validation:")
display(validation_summary)

print(
    "\nOverall Status:",
    "PASS" if (validation_summary["Status"] == "PASS").all()
    else "REVIEW"
)

Final Data Cleaning Validation:


,Dataset,Rows,Missing_Values,Status
0,Customers,10000,0,PASS
1,Policies,15000,0,PASS
2,Claims,25000,0,PASS
3,Payments,15541,0,PASS
4,Risk Assessment,10000,0,PASS
5,Agents,250,0,PASS
6,Policy Agents,15000,0,PASS
7,Vehicles,6000,0,PASS



Overall Status: PASS


## Data Cleaning — Final Conclusion

### Completed Activities

- Standardized date, categorical, and numerical data types.
- Preserved all 96,791 records across eight datasets.
- Created policy-level flags for coverage-date and status discrepancies.
- Identified claims and payments exceeding policy coverage.
- Flagged settled claims without matching payment records.
- Identified claim and payment amount outliers using the IQR method.
- Created a combined claim review flag.
- Validated all eight datasets for record counts and missing values.

### Key Findings

- 7,924 policy-status discrepancies were identified.
- 143 claims and 34 payments exceeded associated policy coverage.
- 732 settled claims had no matching payment records.
- 1,583 claim amount outliers and 1,024 payment amount outliers were identified.
- 2,306 claims (9.22%) were flagged for additional review.
- 189 payments occurred after the September 30, 2026 analysis cutoff.

### Data Cleaning Decisions

- No original records were deleted.
- No financial values or original policy statuses were modified.
- Potential business-rule inconsistencies were retained and flagged.
- Statistical outliers were preserved for further analysis.
- Historical financial reporting will apply the September 30, 2026 cutoff.

### Validation Notes

The claim and payment outlier counts differ slightly from the earlier Data Quality Audit and require reconciliation before final reporting.

### Conclusion

The eight insurance datasets have been standardized and validated for further analysis. All original records were retained, and additional analytical flags were created to support insurance claims, risk, and payment performance analysis.

**Final Validation Status: PASS**